In [1]:
import mne
import numpy as np
from pathlib import Path

BASE = Path(r"C:\Users\KIIT\Desktop\Parkinson_EEG")
RAW = BASE / "data" / "raw"
OUT = BASE / "results1" / "SanDiego"
OUT.mkdir(parents=True, exist_ok=True)

SEG_LEN = 2      # seconds
SF = 256         # assumed sampling rate
BP_LOW, BP_HIGH = 1, 40   # band-pass range

def hjorth_params(sig):
    diff1 = np.diff(sig)
    diff2 = np.diff(diff1)
    v0 = np.var(sig)
    v1 = np.var(diff1)
    v2 = np.var(diff2)

    activity = v0
    mobility = np.sqrt(v1 / v0) if v0 > 0 else 0
    complexity = (np.sqrt(v2 / v1) / mobility) if v1 > 0 and mobility > 0 else 0
    return activity, mobility, complexity


X, y, subjects = [], [], []

files = sorted(RAW.rglob("*.bdf"))
print("Total files:", len(files))

for f in files:
    print("\nProcessing:", f.name)

    raw = mne.io.read_raw_bdf(f, preload=True, verbose=False)

    # --------- CASE-B : Band-pass only (no ICA) ----------
    raw.filter(BP_LOW, BP_HIGH, fir_design="firwin", verbose=False)

    data = raw.get_data()
    n_samples = data.shape[1]
    step = SEG_LEN * SF
    n_segments = n_samples // step

    for i in range(n_segments):
        seg = data[:, i*step:(i+1)*step]

        feats = []
        for ch in seg:
            m = np.mean(ch)
            s = np.std(ch)
            act, mob, comp = hjorth_params(ch)
            feats += [m, s, act, mob, comp]

        X.append(feats)

        label = 1 if "pd" in f.name.lower() else 0
        y.append(label)
        subjects.append(f.stem)

X = np.array(X, float)
y = np.array(y, int)
subjects = np.array(subjects)

print("\nFinal shape:", X.shape)

np.savez(OUT / "features_caseB_Bandpass.npz",
         X=X, y=y, subjects=subjects, fs=SF)

print("\nSaved:", OUT / "features_caseB_Bandpass.npz")

Total files: 46

Processing: sub-hc1_ses-hc_task-rest_eeg.bdf

Processing: sub-hc10_ses-hc_task-rest_eeg.bdf

Processing: sub-hc18_ses-hc_task-rest_eeg.bdf

Processing: sub-hc2_ses-hc_task-rest_eeg.bdf

Processing: sub-hc20_ses-hc_task-rest_eeg.bdf

Processing: sub-hc21_ses-hc_task-rest_eeg.bdf

Processing: sub-hc24_ses-hc_task-rest_eeg.bdf

Processing: sub-hc25_ses-hc_task-rest_eeg.bdf

Processing: sub-hc29_ses-hc_task-rest_eeg.bdf

Processing: sub-hc30_ses-hc_task-rest_eeg.bdf

Processing: sub-hc31_ses-hc_task-rest_eeg.bdf

Processing: sub-hc32_ses-hc_task-rest_eeg.bdf

Processing: sub-hc33_ses-hc_task-rest_eeg.bdf

Processing: sub-hc4_ses-hc_task-rest_eeg.bdf

Processing: sub-hc7_ses-hc_task-rest_eeg.bdf

Processing: sub-hc8_ses-hc_task-rest_eeg.bdf

Processing: sub-pd11_ses-off_task-rest_eeg.bdf

Processing: sub-pd11_ses-on_task-rest_eeg.bdf

Processing: sub-pd12_ses-off_task-rest_eeg.bdf

Processing: sub-pd12_ses-on_task-rest_eeg.bdf

Processing: sub-pd13_ses-off_task-rest_eeg.bdf